In [1]:
import os
import datetime
import logging
import pandas as pd
from pathlib import Path
from delta.tables import DeltaTable
from pyspark.sql import SparkSession, functions as F
import pandas as pd

from ingest_helpers.folder_helpers import get_folder_list, get_dump_dates
from ingest_helpers.config_helpers import load_config_from_json
from ingest_helpers.info_output_helpers import (
    get_or_create_last_log,
    get_last_dump_date,
    output_single_json,
    output_schema_json,
    output_schema_txt,
    output_schema_csv,
)

# from ingest_helpers.spark_processing_helpers2 import build_tree, normalize_tree, sort_tree, select_tree,  explode_nested, split_nested

# from ingest_helpers.spark_processing_helpers2_test import apply_enrich

from ingest_helpers.spark_processing_helpers3 import *


# -----------------------------
# Logging setup
# -----------------------------
LOG_DIR = Path(os.getenv("LOG_DIR", "/logs"))
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = LOG_DIR / "discogs_silver_ingest.log"

logger = logging.getLogger(__name__)
if not logger.handlers:
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
        handlers=[logging.StreamHandler(), logging.FileHandler(LOG_FILE)],
    )

logger.info(f"Logging to {LOG_FILE}")

# -----------------------------
# Environment / config
# -----------------------------
DATA_DIR = os.getenv("DATA_DIR")
BRONZE_DATA_DIR = os.path.join(DATA_DIR, "bronze")
SILVER_INGEST_CONFIG_PATH = os.getenv("SILVER_INGEST_CONFIG_PATH")

OUTPUT_DEBUG_FILES = True

# -----------------------------
# Spark session
# -----------------------------
spark = (
    SparkSession.builder.appName("discogs-silver-ingest")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")


# -----------------------------
# Listing folders & dumps
# -----------------------------
bronze_folders = get_folder_list(BRONZE_DATA_DIR)
dump_dates = get_dump_dates(bronze_folders)

print("")
logger.info(f"{dump_dates = }")
print("")
n = int(os.getenv("NB_DUMPS_TO_PROCESS", 1))
n_latest_dumps_dates = sorted(dump_dates[-n - 1 :])

print("")

logger.info(f"Processing {n} latest dumps: {n_latest_dumps_dates}")
print("")


# Load configs
table_configs = load_config_from_json(SILVER_INGEST_CONFIG_PATH)


# -----------------------------
# Start ingestion
# -----------------------------
previous_input_path = None

for dump_date in n_latest_dumps_dates:
    for cfg in table_configs:
        logger.info(f"")
        logger.info(f"")
        logger.info(
            f"================================================================================================="
        )
        logger.info(f"Processing dump date {dump_date} with config {cfg}")
        logger.info(
            f"================================================================================================="
        )
        print("")

        time_start = datetime.datetime.now()

        layer_input = cfg["layer_input"]
        layer_output = cfg["layer_output"]
        table_input = cfg["table_input"]
        table_output = cfg["table_output"]

        output_path = os.path.join(DATA_DIR, layer_output, table_output)
        input_path = os.path.join(DATA_DIR, layer_input, table_input)

        # if int(dump_date) <= last_processed_dump:
        #    logger.info(f"{dump_date} <= {last_processed_dump}: skipping")
        #    continue

        delta_table_path = os.path.join(
            DATA_DIR, layer_input, table_input, f"dump={dump_date}"
        )
        print("")
        logger.info(f"Loading {layer_input} table: {delta_table_path}")
        logger.info(f"")
        
        df = spark.read.format("delta").load(delta_table_path)

INFO:__main__:Logging to /logs/discogs_silver_ingest.log
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/20 13:36:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
INFO:__main__:dump_dates = ['20251101', '20251201', '20260101']
INFO:__main__:Processing 3 latest dumps: ['20251101', '20251201', '20260101']
INFO:__main__:
INFO:__main__:
INFO:__main__:=================================================================================================
INFO:__main__:Processing dump date 20251101 with config {'layer_input': 'bronze', 'table_input': 'artists', 'layer_output': 'silver', 'table_output': 'artists', 'primary_key': 'id', 'keep': ['data_quality', 'groups', 'id', 'members', 'name', 'realname']}
INFO:__main__:====================================

['/data_tests/bronze', '/data_tests/bronze/artists', '/data_tests/bronze/artists/dump=20251201', '/data_tests/bronze/artists/dump=20251201/_delta_log', '/data_tests/bronze/artists/dump=20251201/_delta_log/_staged_commits', '/data_tests/bronze/artists/dump=20260101', '/data_tests/bronze/artists/dump=20260101/_delta_log', '/data_tests/bronze/artists/dump=20260101/_delta_log/_staged_commits', '/data_tests/bronze/artists/dump=20251101', '/data_tests/bronze/artists/dump=20251101/_delta_log', '/data_tests/bronze/artists/dump=20251101/_delta_log/_staged_commits', '/data_tests/bronze/releases', '/data_tests/bronze/releases/dump=20251201', '/data_tests/bronze/releases/dump=20251201/_delta_log', '/data_tests/bronze/releases/dump=20251201/_delta_log/_staged_commits', '/data_tests/bronze/releases/dump=20260101', '/data_tests/bronze/releases/dump=20260101/_delta_log', '/data_tests/bronze/releases/dump=20260101/_delta_log/_staged_commits', '/data_tests/bronze/releases/dump=20251101', '/data_tests/br

INFO:__main__:
INFO:__main__:
INFO:__main__:=================================================================================================
INFO:__main__:Processing dump date 20251101 with config {'layer_input': 'bronze', 'table_input': 'releases', 'layer_output': 'silver', 'table_output': 'releases', 'primary_key': '@id', 'keep': ['@id', '@status', 'artists', 'extraartists', 'genres', 'released', 'styles', 'title', 'tracklist']}
INFO:__main__:=================================================================================================
INFO:__main__:Loading bronze table: /data_tests/bronze/releases/dump=20251101
INFO:__main__:
INFO:__main__:
INFO:__main__:
INFO:__main__:=================================================================================================
INFO:__main__:Processing dump date 20251201 with config {'layer_input': 'bronze', 'table_input': 'artists', 'layer_output': 'silver', 'table_output': 'artists', 'primary_key': 'id', 'keep': ['data_quality', 'groups', 

In [8]:
df.columns

['@id',
 '@status',
 'artists',
 'companies',
 'country',
 'data_quality',
 'extraartists',
 'formats',
 'genres',
 'identifiers',
 'labels',
 'master_id',
 'notes',
 'released',
 'series',
 'styles',
 'title',
 'tracklist',
 'videos',
 'dump']

In [11]:
df.select("artists").show()

+--------------------+
|             artists|
+--------------------+
|{[{NULL, 1, NULL,...|
|{[{NULL, 2, NULL,...|
|{[{NULL, 3, NULL,...|
|{[{NULL, 21, NULL...|
|{[{NULL, 22, NULL...|
|{[{NULL, 2, NULL,...|
|{[{NULL, 28, NULL...|
|{[{NULL, 29, NULL...|
|{[{NULL, 33, NULL...|
|{[{NULL, 36, NULL...|
|{[{NULL, 33, NULL...|
|{[{NULL, 30, NULL...|
|{[{NULL, 28, NULL...|
|{[{NULL, 50, NULL...|
|{[{NULL, 51, NULL...|
|{[{NULL, 52, NULL...|
|{[{Casy Hogan, 53...|
|{[{NULL, 79947, N...|
|{[{NULL, 55, NULL...|
|{[{NULL, 53, NULL...|
+--------------------+
only showing top 20 rows


In [12]:
from pyspark.sql.functions import explode

df.select("@id", explode("artists.artist")).show()

+---+--------------------+
|@id|                 col|
+---+--------------------+
|  1|{NULL, 1, NULL, T...|
|  2|{NULL, 2, NULL, M...|
|  3|{NULL, 3, NULL, J...|
|  4|{NULL, 21, NULL, ...|
|  5|{NULL, 22, NULL, ...|
|  6|{NULL, 2, NULL, M...|
|  7|{NULL, 28, NULL, ...|
|  8|{NULL, 29, NULL, ...|
|  9|{NULL, 33, NULL, ...|
| 10|{NULL, 36, NULL, ...|
| 11|{NULL, 33, NULL, ...|
| 12|{NULL, 30, NULL, ...|
| 13|{NULL, 28, NULL, ...|
| 14|{NULL, 50, NULL, ...|
| 15|{NULL, 51, NULL, ...|
| 16|{NULL, 52, NULL, ...|
| 17|{Casy Hogan, 53, ...|
| 18|{NULL, 79947, NUL...|
| 19|{NULL, 55, NULL, ...|
| 20|{NULL, 53, NULL, ...|
+---+--------------------+
only showing top 20 rows


In [13]:
df.printSchema()


root
 |-- @id: long (nullable = true)
 |-- @status: string (nullable = true)
 |-- artists: struct (nullable = true)
 |    |-- artist: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- anv: string (nullable = true)
 |    |    |    |-- id: long (nullable = true)
 |    |    |    |-- join: string (nullable = true)
 |    |    |    |-- name: string (nullable = true)
 |-- companies: struct (nullable = true)
 |    |-- company: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- catno: string (nullable = true)
 |    |    |    |-- entity_type: long (nullable = true)
 |    |    |    |-- entity_type_name: string (nullable = true)
 |    |    |    |-- id: long (nullable = true)
 |    |    |    |-- name: string (nullable = true)
 |    |    |    |-- resource_url: string (nullable = true)
 |-- country: string (nullable = true)
 |-- data_quality: string (nullable = true)
 |-- extraartists: struct (nullable = true

In [63]:
df_artists = (df.select(
    "@id",
    explode("artists.artist").alias("artist")
)
    .select("@id", "artist.id", "artist.name")
    .withColumn("track_position",F.lit("All"))
    .withColumn("role",F.lit("Release Artist"))
    .withColumnsRenamed({"id":"artist_id", "name":"artist_name"})
    .withColumnRenamed("@id","release_id")
             )

df_artists.show()

+----------+---------+--------------------+--------------+--------------+
|release_id|artist_id|         artist_name|track_position|          role|
+----------+---------+--------------------+--------------+--------------+
|         1|        1|       The Persuader|           All|Release Artist|
|         2|        2|Mr. James Barth &...|           All|Release Artist|
|         3|        3|           Josh Wink|           All|Release Artist|
|         4|       21|         Faze Action|           All|Release Artist|
|         5|       22|            Datacide|           All|Release Artist|
|         6|        2|Mr. James Barth &...|           All|Release Artist|
|         7|       28|        Moonchildren|           All|Release Artist|
|         8|       29|       Sweet Abraham|           All|Release Artist|
|         9|       33|            Blue Six|           All|Release Artist|
|        10|       36|          Lovetronic|           All|Release Artist|
|        11|       33|            Blue

In [163]:
import pyspark.sql.functions as F

df_extraartists = (
    df
    .select(
        F.col("@id").alias("release_id"),
        F.explode("extraartists.artist").alias("artist")
    )
    .select(
        "release_id",
        F.col("artist.id").alias("artist_id"),
        F.col("artist.name").alias("artist_name"),
        F.col("artist.role"),
        F.col("artist.tracks").alias("track_position")
    ).withColumn(
        "track_position",
        F.when(
            F.col("track_position").isNull(),
            F.lit("All")
        ).otherwise(F.col("track_position"))
    )
    .withColumn("role", F.explode(F.split("role", ", ")))
    .withColumn(
        "track_position",
        F.explode(F.split("track_position", ", "))
    )
    .withColumn("track_title", F.lit("All"))
)

df_extraartists.show()


+----------+---------+--------------------+--------------------+--------------+-----------+
|release_id|artist_id|         artist_name|                role|track_position|track_title|
+----------+---------+--------------------+--------------------+--------------+-----------+
|         1|   507025|George Cutmaster ...|      Lacquer Cut By|           All|        All|
|         1|      239|     Jesper Dahlbäck|Written-By [All T...|           All|        All|
|         2|       26|        Alexi Delano|            Producer|           All|        All|
|         2|       26|        Alexi Delano|         Recorded By|           All|        All|
|         2|       27|      Cari Lekebusch|            Producer|           All|        All|
|         2|       27|      Cari Lekebusch|         Recorded By|           All|        All|
|         2|       26|        Alexi Delano|          Written-By|           All|        All|
|         2|       27|      Cari Lekebusch|          Written-By|           All| 

In [123]:
from pyspark.sql.functions import split


df_track_artists = (df.select(
    "@id",

    explode("tracklist.track").alias("tracks"))
        .select("@id", "tracks.*")
       

)
df_track_artists.show()

+---+--------------------+--------+--------------------+--------+--------------------+
|@id|             artists|duration|        extraartists|position|               title|
+---+--------------------+--------+--------------------+--------+--------------------+
|  1|                NULL|    4:45|                NULL|       A|           Östermalm|
|  1|                NULL|    6:11|                NULL|      B1|          Vasastaden|
|  1|                NULL|    2:49|                NULL|      B2|         Kungsholmen|
|  1|                NULL|    5:38|                NULL|      C1|           Södermalm|
|  1|                NULL|    4:52|                NULL|      C2|            Norrmalm|
|  1|                NULL|    5:16|                NULL|       D|          Gamla Stan|
|  2|                NULL|    5:08|                NULL|      A1|         A Sea Apart|
|  2|                NULL|    4:21|                NULL|      A2|         Dutchmaster|
|  2|                NULL|    4:22|        

In [91]:

df_tracks_artists = (df.select(
    "@id",

    explode("tracklist.track")
        .alias("tracks"))
        .select("@id", explode("tracks.artists.artist")
            .alias("artists"),"tracks.position","tracks.title")
    .select("@id","position","title","artists.id","artists.name")
    .withColumn("role",F.lit("Track artist"))
    .withColumnsRenamed({"id":"artist_id","title":"track_title","position":"track_position", "name":"artist_name"})
    .withColumnRenamed("@id","release_id")
    
       

)
df_tracks_artists.show()

+----------+--------------+--------------------+---------+--------------------+------------+
|release_id|track_position|         track_title|artist_id|         artist_name|        role|
+----------+--------------+--------------------+---------+--------------------+------------+
|         3|             1|                  D2|        4|       Johannes Heil|Track artist|
|         3|             1|                  D2|        5|          Heiko Laux|Track artist|
|         3|             2|    Anjua (Sneaky 3)|    15525|   Karl Axel Bissler|Track artist|
|         3|             3|When The Funk Hit...|        7|            Sylk 130|Track artist|
|         3|             4|What's The Time, ...|        1|       The Persuader|Track artist|
|         3|             5|              Vol. 2|   267132|    Care Company (2)|Track artist|
|         3|             6|  Political Prisoner|     6981|          Gez Varley|Track artist|
|         3|             7|         Pop Kulture|       11|            

In [161]:

df_tracks_extraartists = (df.select(
    "@id",

    explode("tracklist.track")
        .alias("tracks"))
        .select("@id", explode("tracks.extraartists.artist")
            .alias("artists"),"tracks.position","tracks.title")
    .select("@id","position","title","artists.id","artists.name","artists.role")

    
    .withColumnsRenamed({"id":"artist_id","title":"track_title","position":"track_position", "name":"artist_name"})
    .withColumnRenamed("@id","release_id").select(
    "release_id",
    "artist_id",
    "artist_name",
    "track_position",
    "track_title",
    explode(split("role", ", ")).alias("role"))
                  )


df_tracks_extraartists.show()

+----------+---------+------------------+--------------+--------------------+--------------+
|release_id|artist_id|       artist_name|track_position|         track_title|          role|
+----------+---------+------------------+--------------+--------------------+--------------+
|         3|        8|     Mood II Swing|             3|When The Funk Hit...|         Remix|
|         3|       23|        Alex Hi-Fi|             8|K-Mart Shopping (...|         Remix|
|         3|       14|  Eight Miles High|             9|Lovelee Dae (Eigh...|         Remix|
|         3|    67226|     Stacey Pullen|            10|               Sweat|     Presenter|
|         4|   246316|         Raj Gupta|             2|  To Love Is To Grow|    Written-By|
|         4|    56668|      Zeke Manyika|             2|  To Love Is To Grow|    Written-By|
|         4|     9559|   Vanessa Freeman|             4|           Heartbeat|    Written-By|
|         4|    56668|      Zeke Manyika|             6|   Got To Find

In [156]:
df_artists_all =( 
    df_tracks_extraartists
    .unionByName(df_tracks_artists, allowMissingColumns=True)    
        .unionByName(df_extraartists, allowMissingColumns=True)    
        .unionByName(df_artists, allowMissingColumns=True).
        distinct()
                )

df_artists_all.show()

+----------+---------+--------------------+--------------+--------------------+--------------------+
|release_id|artist_id|         artist_name|track_position|         track_title|                role|
+----------+---------+--------------------+--------------+--------------------+--------------------+
|        37|    12433|Love From San Fra...|             5|165 Drop (Love Fr...|               Remix|
|       102|  1147165|   Randy Johnson (3)|            A2|Why U Wanna? (Ale...|              Vocals|
|       128|   476951|       Pete Williams|            A2|Jive (Natural Rhy...|            Music By|
|       237|   168579|            Nico (4)|            A1|        Killamanjaro|            Mixed By|
|       251|      278|              Peshay|             A|Music (Peshay Rew...|      Remix [Rework]|
|       297|      391|          David Toop|             2|Let All Mortal Fl...|         Arranged By|
|       320|      466|                Plug|           1-1|         Lunar Laugh|            